In [ ]:
# Data Exploration Notebook
# Run this in Jupyter: jupyter notebook notebooks/explore_data.ipynb

import numpy as np
import matplotlib.pyplot as plt
import pickle
import sys
sys.path.append('../src')
from dataset import DeforestationDataset

# Load a dataset to explore
data_dir = "../data/processed"
metadata_file = f"{data_dir}/train_metadata.pkl"
norm_stats_file = f"{data_dir}/normalization_stats.pkl"

dataset = DeforestationDataset(data_dir, metadata_file, norm_stats_file, augment=False)

print(f"Dataset size: {len(dataset)}")

# Load sample data
sample_chip, sample_mask = dataset[0]
print(f"Chip shape: {sample_chip.shape}")
print(f"Mask shape: {sample_mask.shape}")
print(f"Chip data type: {sample_chip.dtype}")
print(f"Mask data type: {sample_mask.dtype}")

def plot_sample(chip, mask, title="Sample"):
    """Plot a sample with RGB, NDVI, dNBR, and mask."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Assuming band order: [B2, B3, B4, B8, B11, B12, NDVI_pre, NBR_pre, 
    #                       B2, B3, B4, B8, B11, B12, NDVI_post, NBR_post, dNDVI, dNBR]
    
    # RGB composite (using post-deforestation bands)
    # Adjust indices based on your actual band order
    rgb_bands = [11, 10, 9]  # B4, B3, B2 from post period
    rgb = chip[rgb_bands].permute(1, 2, 0).numpy()
    rgb = np.clip((rgb - rgb.min()) / (rgb.max() - rgb.min()), 0, 1)
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title('RGB (Post)')
    axes[0, 0].axis('off')
    
    # NDVI pre
    ndvi_pre = chip[6].numpy()  # Adjust index
    im1 = axes[0, 1].imshow(ndvi_pre, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[0, 1].set_title('NDVI (Pre)')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)
    
    # NDVI post
    ndvi_post = chip[14].numpy()  # Adjust index
    im2 = axes[0, 2].imshow(ndvi_post, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[0, 2].set_title('NDVI (Post)')
    axes[0, 2].axis('off')
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)
    
    # dNDVI
    dndvi = chip[18].numpy()  # Adjust index
    im3 = axes[1, 0].imshow(dndvi, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[1, 0].set_title('dNDVI (Change)')
    axes[1, 0].axis('off')
    plt.colorbar(im3, ax=axes[1, 0], fraction=0.046)
    
    # dNBR
    dnbr = chip[19].numpy()  # Adjust index
    im4 = axes[1, 1].imshow(dnbr, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[1, 1].set_title('dNBR (Change)')
    axes[1, 1].axis('off')
    plt.colorbar(im4, ax=axes[1, 1], fraction=0.046)
    
    # Mask
    mask_np = mask[0].numpy()
    im5 = axes[1, 2].imshow(mask_np, cmap='Reds', vmin=0, vmax=1)
    axes[1, 2].set_title('Deforestation Mask')
    axes[1, 2].axis('off')
    plt.colorbar(im5, ax=axes[1, 2], fraction=0.046)
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

# Plot a few samples
for i in range(3):
    chip, mask = dataset[i]
    has_defor = mask.sum() > 0
    plot_sample(chip, mask, f"Sample {i} (Deforestation: {has_defor})")

# Plot statistics
def plot_band_statistics():
    """Plot histograms of band values."""
    # Sample 20 chips for statistics
    n_samples = min(20, len(dataset))
    all_chips = []
    all_masks = []
    
    for i in range(n_samples):
        chip, mask = dataset[i]
        all_chips.append(chip.numpy())
        all_masks.append(mask.numpy())
    
    all_chips = np.stack(all_chips)  # (n_samples, n_bands, H, W)
    all_masks = np.stack(all_masks)  # (n_samples, 1, H, W)
    
    n_bands = all_chips.shape[1]
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.flatten()
    
    band_names = ['B2_pre', 'B3_pre', 'B4_pre', 'B8_pre', 'B11_pre', 'B12_pre', 
                  'NDVI_pre', 'NBR_pre', 'B2_post', 'B3_post', 'B4_post', 'B8_post', 
                  'B11_post', 'B12_post', 'NDVI_post', 'NBR_post', 'dNDVI', 'dNBR']
    
    for i in range(min(n_bands, len(axes))):
        band_data = all_chips[:, i, :, :].flatten()
        axes[i].hist(band_data, bins=50, alpha=0.7)
        axes[i].set_title(f'{band_names[i] if i < len(band_names) else f"Band {i}"}')
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')
    
    # Hide unused subplots
    for i in range(n_bands, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Mask statistics
    mask_coverage = np.mean(all_masks > 0, axis=(2, 3))  # Fraction of pixels with deforestation
    plt.figure(figsize=(10, 6))
    plt.hist(mask_coverage.flatten(), bins=20)
    plt.xlabel('Fraction of Pixels with Deforestation')
    plt.ylabel('Number of Chips')
    plt.title('Distribution of Deforestation Coverage per Chip')
    plt.show()

plot_band_statistics()

# Check for potential issues
def check_data_quality():
    """Check for common data issues."""
    issues = []
    
    # Check a sample of data
    n_check = min(50, len(dataset))
    
    for i in range(n_check):
        chip, mask = dataset[i]
        
        # Check for NaN or inf values
        if torch.isnan(chip).any() or torch.isinf(chip).any():
            issues.append(f"Sample {i}: NaN/Inf in chip")
        
        if torch.isnan(mask).any() or torch.isinf(mask).any():
            issues.append(f"Sample {i}: NaN/Inf in mask")
        
        # Check mask values are in [0, 1]
        if mask.min() < 0 or mask.max() > 1:
            issues.append(f"Sample {i}: Mask values outside [0, 1]")
        
        # Check for extremely large values
        if chip.abs().max() > 100:
            issues.append(f"Sample {i}: Very large chip values (max: {chip.abs().max()})")
    
    if issues:
        print("Data quality issues found:")
        for issue in issues[:10]:  # Show first 10 issues
            print(f"  {issue}")
        if len(issues) > 10:
            print(f"  ... and {len(issues) - 10} more issues")
    else:
        print("No data quality issues found!")

check_data_quality()

print("\nExploration complete! Review the plots and statistics above.")
print("Key things to check:")
print("1. Do the RGB images look reasonable?")
print("2. Do the change indices (dNDVI, dNBR) show clear differences where there's deforestation?")
print("3. Are the masks aligned with visible changes in the imagery?")
print("4. Are the band statistics reasonable (no extreme outliers)?")